# OMBRIA Cloud Pilot

Run this notebook on Colab or Kaggle with GPU enabled. It does not store Google credentials or private tokens.

In [ ]:
!python --version
!nvidia-smi || true

## Clone Repositories

Replace `YOUR_GITHUB_REPO_URL` after this project is pushed to GitHub. Until then, upload the project folder or run these commands locally in a cloud terminal.

In [ ]:
# !git clone YOUR_GITHUB_REPO_URL geoai-paper
# %cd geoai-paper
!mkdir -p external
!test -d external/OMBRIA || git clone --depth 1 https://github.com/geodrak/OMBRIA.git external/OMBRIA

In [ ]:
!pip install -q numpy pillow torch torchvision

## Dry Run

In [ ]:
!python scripts/train_ombria_unet.py --root external/OMBRIA --variant multimodal --dry-run

## Short Clean-Variant Runs

Use 5 epochs first. If the metrics move and runtime is acceptable, increase to 30 epochs.

In [ ]:
%%bash
set -e
for variant in s1_after s1_bitemporal s2_after s2_bitemporal multimodal; do
  python scripts/train_ombria_unet.py \
    --root external/OMBRIA \
    --variant "$variant" \
    --epochs 5 \
    --batch-size 8 \
    --base-channels 16
done

## Multimodal Degradation Evaluation

This evaluates the same clean multimodal checkpoint under synthetic S2 degradation.

In [ ]:
%%bash
set -e
for mode in none zero_after zero_all noise_after patch_after; do
  python scripts/train_ombria_unet.py \
    --root external/OMBRIA \
    --variant multimodal \
    --degrade-s2 "$mode" \
    --batch-size 8 \
    --base-channels 16 \
    --eval-checkpoint results/runs/ombria/multimodal_none_seed7/best_model.pt
done

## Summarize

In [ ]:
!python scripts/summarize_ombria_runs.py --runs-dir results/runs/ombria --out results/tables/ombria_run_summary.csv
!cat results/tables/ombria_run_summary.csv